In [1]:
# -*- coding: utf-8 -*-
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt
import autokeras as ak

# Importar ferramentas de avaliação do scikit-learn (ainda úteis para análise)
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# --- Parâmetros de Configuração ---
# Estas dimensões devem corresponder EXATAMENTE à saída da sua STFT.
# Verifique com um arquivo .npy: np.load('caminho/do/seu/arquivo.npy').shape
IMG_HEIGHT = 33 # Número de bins de frequência (rows de Zxx)
IMG_WIDTH = 81 # Número de bins de tempo (columns de Zxx)
NUM_CLASSES = 3 # 'rest', 'left', 'right'
BATCH_SIZE = 16 # Pode ser otimizado pelo AutoKeras, mas vamos manter um valor inicial
MAX_TRIALS = 10 # Número máximo de diferentes modelos que o AutoKeras vai tentar
EPOCHS_PER_TRIAL = 50 # Número de épocas para treinar cada modelo experimental do AutoKeras

# Mapeamento de rótulos para inteiros
label_map = {'rest': 0, 'left': 1, 'right': 2}
class_names = ['rest', 'left', 'right']

# --- Função para carregar e pré-processar um único espectrograma (.npy) ---
# Esta função é a mesma.
def load_spectrogram_data(filepath_c3, filepath_c4, label):
    spectrogram_c3 = np.load(filepath_c3.numpy()).astype(np.float32)
    spectrogram_c4 = np.load(filepath_c4.numpy()).astype(np.float32)

    min_c3, max_c3 = np.min(spectrogram_c3), np.max(spectrogram_c3)
    spectrogram_c3 = (spectrogram_c3 - min_c3) / (max_c3 - min_c3 + 1e-8) if (max_c3 - min_c3) > 1e-8 else spectrogram_c3

    min_c4, max_c4 = np.min(spectrogram_c4), np.max(spectrogram_c4)
    spectrogram_c4 = (spectrogram_c4 - min_c4) / (max_c4 - min_c4 + 1e-8) if (max_c4 - min_c4) > 1e-8 else spectrogram_c4

    combined_spectrogram = np.stack([spectrogram_c3, spectrogram_c4], axis=-1)
    
    return combined_spectrogram, label

# --- Função para Criar o Dataset (Adaptada para AutoKeras) ---
# A função é a mesma, mas agora ela pode retornar o dataset completo
# ou apenas os dados e rótulos como arrays NumPy se preferir.
def create_dataset_from_directories_npy(data_root_dir, is_training_data=True):
    all_filepaths_c3 = []
    all_filepaths_c4 = []
    all_labels = []

    print(f"Tentando ler do diretório raiz: {data_root_dir}")
    if not os.path.exists(data_root_dir):
        print(f"Erro: O diretório raiz dos dados não existe: {data_root_dir}")
        return None, None
            
    for class_name in label_map.keys():
        class_dir = os.path.join(data_root_dir, class_name)
        # print(f"  Verificando diretório da classe: {class_dir}") # Descomente para depuração
        if not os.path.exists(class_dir):
            print(f"  Aviso: O diretório da classe '{class_name}' não existe em {data_root_dir}. Pulando.")
            continue 

        for img_filename in os.listdir(class_dir):
            if img_filename.endswith('_C3.npy'):
                filepath_c3 = os.path.join(class_dir, img_filename)
                filepath_c4 = filepath_c3.replace('_C3.npy', '_C4.npy')
                
                if os.path.exists(filepath_c4):
                    all_filepaths_c3.append(filepath_c3)
                    all_filepaths_c4.append(filepath_c4)
                    all_labels.append(label_map[class_name])
                # else:
                    # print(f"    Aviso: Arquivo C4 correspondente para {filepath_c3} não encontrado. Pulando este par.") # Descomente para depuração
    
    print(f"Total de arquivos C3 encontrados: {len(all_filepaths_c3)}")
    print(f"Total de arquivos C4 encontrados: {len(all_filepaths_c4)}")
    print(f"Total de rótulos encontrados: {len(all_labels)}")

    if len(all_filepaths_c3) == 0:
        print("Erro: Nenhum par de espectrogramas C3/C4 encontrado. O dataset está vazio.")
        return None, None

    filepaths_c3_tensor = tf.constant(all_filepaths_c3, dtype=tf.string)
    filepaths_c4_tensor = tf.constant(all_filepaths_c4, dtype=tf.string)
    labels_tensor = tf.constant(all_labels, dtype=tf.int32)

    dataset = tf.data.Dataset.from_tensor_slices(((filepaths_c3_tensor, filepaths_c4_tensor), labels_tensor))
    dataset = dataset.map(lambda x, y: tf.py_function(load_spectrogram_data, [x[0], x[1], y], (tf.float32, tf.int32)),
                          num_parallel_calls=tf.data.AUTOTUNE)
    
    dataset = dataset.map(lambda spec, label: (tf.ensure_shape(spec, (IMG_HEIGHT, IMG_WIDTH, 2)), tf.ensure_shape(label, ())),
                          num_parallel_calls=tf.data.AUTOTUNE)

    # AutoKeras pode lidar com o shuffling e batching internamente,
    # mas pré-batching para dataset grande é bom
    if is_training_data:
        dataset_size = len(all_filepaths_c3)
        train_size = int(0.8 * dataset_size)
        dataset = dataset.shuffle(buffer_size=dataset_size) # Embaralha o dataset completo
        train_ds = dataset.take(train_size)
        val_ds_split = dataset.skip(train_size) # Esta é a validação interna do AutoKeras
        return train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE), \
               val_ds_split.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    else:
        # Para o dataset de validação externo (teste)
        return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


# --- Aplicação do AutoKeras ---
if __name__ == '__main__':
    # Escolha a duração da janela para a qual você deseja treinar e avaliar com AutoKeras
    selected_window_ms = 200 # <-- AJUSTE ISSO para a janela desejada

    # Caminhos para os datasets de treinamento e validação
    train_data_root_dir = f'unified_spectrograms_{selected_window_ms}ms'
    val_data_root_dir = f'validation_unified_spectrograms_{selected_window_ms}ms'

    print(f"\n--- Aplicando AutoKeras para a janela de {selected_window_ms}ms ---")

    # Carregar os datasets de treinamento e validação para o AutoKeras
    # Nota: AutoKeras pode pegar tf.data.Dataset diretamente.
    train_ds, ak_val_ds = create_dataset_from_directories_npy(train_data_root_dir, is_training_data=True)
    
    # Carregar o dataset de validação EXTERNO para avaliação final do melhor modelo
    # Este dataset NÃO DEVE ser visto pelo AutoKeras durante a busca (apesar do nome 'val_ds_split' acima)
    # Ele será usado para a avaliação final do melhor modelo encontrado.
    final_val_ds = create_dataset_from_directories_npy(val_data_root_dir, is_training_data=False)

    if train_ds is None or tf.data.experimental.cardinality(train_ds).numpy() == 0:
        print(f"Não há dados de treinamento suficientes em {train_data_root_dir}. AutoKeras abortado.")
    elif final_val_ds is None or tf.data.experimental.cardinality(final_val_ds).numpy() == 0:
        print(f"Não há dados de validação suficientes em {val_data_root_dir}. AutoKeras abortado.")
    else:
        print(f"Dados de treinamento carregados de: {train_data_root_dir}")
        print(f"Dados de validação (interna AutoKeras) carregados de: {train_data_root_dir} (split)")
        print(f"Dados de validação (externa final) carregados de: {val_data_root_dir}")

        # Inicializar o ImageClassifier do AutoKeras
        # 'objective' define a métrica a ser otimizada (default é 'val_accuracy')
        # 'max_trials' é o número de diferentes arquiteturas que o AutoKeras tentará
        # 'overwrite' True para recomeçar a busca, False para continuar de um checkpoint (se houver)
        clf = ak.ImageClassifier(
            overwrite=True,
            max_trials=MAX_TRIALS, # Quantas arquiteturas diferentes testar
            objective="val_accuracy",
            directory="autokeras_models", # Onde salvar os resultados da busca
            project_name=f"spectrogram_classifier_{selected_window_ms}ms" # Nome do projeto
        )

        # Treinar o classificador
        # AutoKeras vai procurar a melhor arquitetura e treinar modelos para cada tentativa
        print("\n--- Iniciando a busca da melhor arquitetura com AutoKeras ---")
        clf.fit(
            train_ds,
            validation_data=ak_val_ds, # Usar a parte do dataset de treino para validação interna do AutoKeras
            epochs=EPOCHS_PER_TRIAL # Quantas épocas cada "trial" será treinado
        )

        # Obter o melhor modelo encontrado pelo AutoKeras
        best_model = clf.export_model()
        
        print("\n--- Melhor Modelo Encontrado pelo AutoKeras ---")
        best_model.summary()

        # Avaliar o melhor modelo no dataset de validação externo (totalmente não visto)
        print("\n--- Avaliação Final do Melhor Modelo no Dataset de Validação Externo ---")
        loss, accuracy = best_model.evaluate(final_val_ds, verbose=0)
        print(f"\nResultados da Avaliação Final ({selected_window_ms}ms):")
        print(f"  Perda no conjunto de validação externo: {loss:.4f}")
        print(f"  Acurácia no conjunto de validação externo: {accuracy*100:.2f}%")

        # Opcional: Salvar o melhor modelo encontrado pelo AutoKeras
        best_model_save_path = f'autokeras_best_model_{selected_window_ms}ms.keras'
        best_model.save(best_model_save_path)
        print(f"Melhor modelo AutoKeras salvo como '{best_model_save_path}'")

        # --- Coletar Previsões para Matriz de Confusão e Relatório ---
        print("\n--- Coletando previsões para análise detalhada no dataset de validação externo ---")
        all_predictions = []
        all_true_labels = []

        for spectrograms_batch, labels_batch in final_val_ds:
            predictions_batch = best_model.predict(spectrograms_batch, verbose=0)
            predicted_classes_batch = np.argmax(predictions_batch, axis=1)
            
            all_predictions.extend(predicted_classes_batch)
            all_true_labels.extend(labels_batch.numpy())
        
        all_predictions = np.array(all_predictions)
        all_true_labels = np.array(all_true_labels)

        # 6. Gerar e Plotar Matriz de Confusão
        print("\n--- Gerando Matriz de Confusão ---")
        cm = confusion_matrix(all_true_labels, all_predictions)
        print(cm)

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=class_names, yticklabels=class_names)
        plt.title(f'Matriz de Confusão do AutoKeras ({selected_window_ms}ms)')
        plt.xlabel('Rótulo Predito')
        plt.ylabel('Rótulo Verdadeiro')
        plt.show()

        # 7. Gerar Relatório de Classificação
        print("\n--- Relatório de Classificação ---")
        report = classification_report(all_true_labels, all_predictions, target_names=class_names, zero_division=0)
        print(report)

        # 8. Visualizar alguns Espectrogramas com Previsões
        print("\n--- Visualizando alguns espectrogramas com previsões (C3 e C4) do melhor modelo AutoKeras ---")
        num_samples_to_plot = 5 # Quantos exemplos você quer visualizar
        
        # Reiniciar o dataset para pegar as primeiras amostras
        final_val_ds_for_plot = create_dataset_from_directories_npy(val_data_root_dir, is_training_data=False)
        
        fig, axes = plt.subplots(num_samples_to_plot, 2, figsize=(14, 4 * num_samples_to_plot), sharex=True)
        axes = axes.flatten()

        sample_count = 0
        for spectrograms_batch, labels_batch in final_val_ds_for_plot.take(num_samples_to_plot // BATCH_SIZE + 1):
            if sample_count >= num_samples_to_plot:
                break
            
            predictions_batch = best_model.predict(spectrograms_batch, verbose=0)

            for i in range(min(BATCH_SIZE, num_samples_to_plot - sample_count)):
                spec_c3 = spectrograms_batch[i,:,:,0].numpy().T
                spec_c4 = spectrograms_batch[i,:,:,1].numpy().T
                
                true_label_idx = labels_batch[i].numpy()
                predicted_label_idx = np.argmax(predictions_batch[i])

                ax_c3 = axes[sample_count * 2]
                ax_c3.imshow(spec_c3, aspect='auto', origin='lower', cmap='viridis')
                ax_c3.set_title(f'C3 - Real: {class_names[true_label_idx]} Pred: {class_names[predicted_label_idx]}')
                ax_c3.set_ylabel('Frequência [Hz]')
                if sample_count == num_samples_to_plot - 1:
                    ax_c3.set_xlabel('Tempo [s]')
                ax_c3.set_ylim([8, 30])

                ax_c4 = axes[sample_count * 2 + 1]
                ax_c4.imshow(spec_c4, aspect='auto', origin='lower', cmap='viridis')
                ax_c4.set_title(f'C4 - Real: {class_names[true_label_idx]} Pred: {class_names[predicted_label_idx]}')
                if sample_count == num_samples_to_plot - 1:
                    ax_c4.set_xlabel('Tempo [s]')
                ax_c4.set_ylim([8, 30])

                sample_count += 1
                if sample_count >= num_samples_to_plot:
                    break
            
        plt.tight_layout()
        plt.show()

c:\Codes\EEG_Study\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



--- Aplicando AutoKeras para a janela de 200ms ---
Tentando ler do diretório raiz: unified_spectrograms_200ms
Total de arquivos C3 encontrados: 4500
Total de arquivos C4 encontrados: 4500
Total de rótulos encontrados: 4500
Tentando ler do diretório raiz: validation_unified_spectrograms_200ms
Total de arquivos C3 encontrados: 2340
Total de arquivos C4 encontrados: 2340
Total de rótulos encontrados: 2340
Dados de treinamento carregados de: unified_spectrograms_200ms
Dados de validação (interna AutoKeras) carregados de: unified_spectrograms_200ms (split)
Dados de validação (externa final) carregados de: validation_unified_spectrograms_200ms


--- Iniciando a busca da melhor arquitetura com AutoKeras ---


Search: Running Trial #1

Value             |Best Value So Far |Hyperparameter
vanilla           |vanilla           |image_block_1/block_type
True              |True              |image_block_1/normalize
False             |False             |image_block_1/augment
3                 |3   

c:\Codes\EEG_Study\venv\lib\site-packages\keras\src\models\functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor']
Received: inputs=Tensor(shape=(None, 33, 81, 2))
  warnings.warn(msg)


225/225 ━━━━━━━━━━━━━━━━━━━━ 39s 118ms/step - accuracy: 0.4828 - loss: 1.0939 - val_accuracy: 0.4944 - val_loss: 1.0468
Epoch 2/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 36s 117ms/step - accuracy: 0.5016 - loss: 1.0457 - val_accuracy: 0.4878 - val_loss: 1.0474
Epoch 3/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 39s 126ms/step - accuracy: 0.4969 - loss: 1.0459 - val_accuracy: 0.5200 - val_loss: 1.0244
Epoch 4/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 42s 131ms/step - accuracy: 0.5096 - loss: 1.0404 - val_accuracy: 0.4989 - val_loss: 1.0416
Epoch 5/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 42s 133ms/step - accuracy: 0.4885 - loss: 1.0518 - val_accuracy: 0.5067 - val_loss: 1.0332
Epoch 6/50


KeyboardInterrupt: 